## __Tópicos avanzados en Inteligencia Artificial 1 - MIA__

__Profesor__: Anthony D. Cho

__Ayudante__: Luis Oliveros

**Asunto**: Autoencoder Variacional
*****

__Basado en la implementación:__ [Francesco Franco](https://medium.com/h7w/implementing-a-variational-autoencoder-with-keras-e19d7140ad90)

## Librerias

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from time import time
from numpy import prod, newaxis, zeros, linspace, array, arange, round, shape
from numpy.random import seed, randint
from collections import Counter
from itertools import permutations
import matplotlib.pyplot as plt
%matplotlib inline

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, metrics
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.optimizers import Adam


### Funciones personalizadas

In [ ]:
def plot_history(history, width=12, height=6):
  """
  DESCRIPTION:
    History performance of the keras model
  
  INPUT:
    @param history: history of performance of fitted model
    @type history: tensorflow.python.keras.callbacks.History

  OUTPUT:
    A graphic
  """

  ## Metrics keys stored in tensorflow object
  keys = list(history.history.keys())

  ## Number of epoch used for fit the model
  epoch = range(1, len(history.epoch) +1)

  ## Check if validation set was used.
  withValidation = False
  for key in keys:
    if 'val' in key:
      withValidation = True

  ## Number of metrics 
  nMetrics = len(keys)
  if withValidation:
    nMetrics = nMetrics//2

  ## Plot-space instance
  plt.figure(figsize=(width, height))

  for i in range(nMetrics):
    plt.subplot(nMetrics, 1, i+1)

    ## Plot (train) metric value
    labelMetric = keys[i]
    metric = history.history[keys[i]]
    plt.plot(epoch, metric, 'o-', label=labelMetric)

    if withValidation:
      ## Plot (validation) metric value
      labelMetricVal = keys[i+nMetrics]
      metricVal = history.history[keys[i+nMetrics]]
      plt.plot(epoch, metricVal, 'o-', label=labelMetricVal)

    plt.xlim(epoch[0], epoch[-1])
    plt.legend()
    plt.grid()

  plt.xlabel('Epoch')
  plt.show()

## Dataset

<center>
    <img src=https://upload.wikimedia.org/wikipedia/commons/f/f7/MnistExamplesModified.png width=800>
</center>

MNIST (Modified National Institute of Standards and Technology) es una gran base de datos de dígitos escritos a mano que se utiliza habitualmente para entrenar diversos sistemas de procesamiento de imágenes. La base de datos contiene 60.000 imágenes de entrenamiento y 10.000 imágenes de prueba.

**Objetivo**: Clasificar las imágenes según su dígito. 


#### Carga de datos

In [ ]:
## Load MNIST dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

## Add a channel to the dataset
X_train = X_train[:, :, :, newaxis]
X_test = X_test[:, :, :, newaxis]

print('Train (shape) X: {}, y: {}'.format(X_train.shape, y_train.shape))
print('Train (shape) X: {}, y: {}'.format(X_test.shape, y_test.shape))

## Mostrar distribución de las clases para cada conjunto.
display(Counter(y_train))
Counter(y_test)

## Preprocesamiento de datos

In [ ]:
## scaling
X_train = X_train / 255
X_test = X_test / 255

### Diseño del modelo

In [ ]:
class VAE(Model):

    def __init__(self, image_width, image_height, num_channels, latent_dim=2):
        """
            DESCRIPTION:
                VAE constructor definition

            INPUT:
                @param image_width: image width used
                @type image_width: integer

                @param image_height: image height used
                @type image_height: integer

                @param num_channels: number of channels (deep) of a image. 
                @type num_channels: integer

                @param latent_dim: latent space dimension
                @type latent_dim: integer
        """

        super(VAE, self).__init__()
        self.latent_dim = latent_dim
        self.img_width = image_width
        self.img_height = image_height
        self.num_channels = num_channels
        self.shape_before_flattening = None
        
        # Create encoder
        self.encoder = self._build_encoder()
        
        # Create decoder
        self.decoder = self._build_decoder()
        
        # Metrics
        self.total_loss_tracker = metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = metrics.Mean(name="kl_loss")
        
    def _build_encoder(self):
        """
            Encoder Architecture using convolutional layers
        """

        encoder_inputs = layers.Input(shape=(self.img_height, self.img_width, self.num_channels), 
                                      name='Encoder_input')
        x = layers.Conv2D(filters=8, kernel_size=3, strides=2, 
                          padding='same', activation='relu', 
                          name='Encoder_Conv2D_01')(encoder_inputs)
        x = layers.BatchNormalization(name='Encoder_BN_01')(x)
        x = layers.Conv2D(filters=16, kernel_size=3, strides=2, 
                          padding='same', activation='relu', 
                          name='Encoder_Conv2D_02')(x)
        x = layers.BatchNormalization(name='Encoder_BN_02')(x)

        # Save shape of the last feature map for the decoder
        self.shape_before_flattening = tuple(x.shape[1:])

        x = layers.Flatten(name='Encoder_Flatten')(x)
        x = layers.Dense(units=20, activation='relu', 
                         name='Encoder_dense')(x)
        x = layers.BatchNormalization(name='Encoder_BN_03')(x)
        
        ## Latent vectors
        z_mean = layers.Dense(units=self.latent_dim, 
                              name='z_mean')(x)
        z_log_var = layers.Dense(units=self.latent_dim, 
                                 name='z_log_var')(x)
        
        ## Latent layer: Sample z
        z = layers.Lambda(self.sampling, name='z')([z_mean, z_log_var])
        
        ## Encoder model builder
        encoder = Model(inputs=encoder_inputs, 
                        outputs=[z_mean, z_log_var, z], 
                        name='encoder')

        ## return model 
        return encoder
    
    def _build_decoder(self):
        """
            Decoder Architecture using convolutional transpose layers
        """

        decoder_inputs = layers.Input(shape=(self.latent_dim,), 
                                      name='Decoder_Input')        
        x = layers.Dense(prod(self.shape_before_flattening), activation='relu',
                         name='Decoder_Dense_01')(decoder_inputs)
        x = layers.BatchNormalization(name='Decoder_BN_01')(x)
        
        ## Reshape to match encoder output before flattening
        x = layers.Reshape(self.shape_before_flattening, 
                           name='Decoder_Reshape')(x)  
        
        x = layers.Conv2DTranspose(filters=16, kernel_size=3, strides=2, 
                                   padding='same', activation='relu', name='Decoder_Conv2DT_01')(x)
        x = layers.BatchNormalization(name='Decoder_BN_02')(x)
        x = layers.Conv2DTranspose(filters=8, kernel_size=3, strides=2, 
                                   padding='same', activation='relu', name='Decoder_Conv2DT_02')(x)
        x = layers.BatchNormalization(name='Decoder_BN_03')(x)
        
        decoder_outputs = layers.Conv2DTranspose(filters=self.num_channels, kernel_size=3, 
                                                 padding='same', activation='sigmoid', 
                                                 name='Decorder_output')(x)
        
        ## Decoder model builder
        decoder = Model(inputs=decoder_inputs, 
                        outputs=decoder_outputs, 
                        name='decoder')
        
        ## return model
        return decoder
    
    def sampling(self, args):
        """
            Function in charge of random sampling in the Gaussian latent space.
        """
        z_mean, z_log_var = args
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]

        ## Sampling from a standard normal
        epsilon = tf.random.normal(shape=(batch, dim))

        ## Return uncentered sampling given by mean and std vectors
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon
    
    def call(self, inputs):
        """
            Function in charge of the model operations flow.
        """
        ## Forward pass through the model

        ## Evaluate encoder model
        z_mean, z_log_var, z = self.encoder(inputs)

        ## Evaluate decoder model
        reconstructed = self.decoder(z)

        ## return output
        return reconstructed
    
    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]
    
    def train_step(self, data):
        # In train_step, data can be either a tensor or a tuple of two tensors
        # If it's a tuple, we're expecting data and targets, but with VAE we don't need targets
        # so we just take the first element
        if isinstance(data, tuple):
            data = data[0]
            
        with tf.GradientTape() as tape:

            # Forward pass
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            
            # Calculate cumulated reconstruction loss
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    binary_crossentropy(data, reconstruction),
                    axis=(1, 2)
                )
            ) #* self.img_width * self.img_height
            
            # Calculate KL divergence loss - https://mbernste.github.io/posts/vae/
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                    axis=1
                )
            )
            
            # Total loss
            total_loss = reconstruction_loss + kl_loss
        
        # Compute gradients and update weights
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        
        # Update metrics
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }
    
    def test_step(self, data):
        # In test_step, data can be either a tensor or a tuple of two tensors
        # If it's a tuple, we're expecting data and targets, but with VAE we don't need targets
        # so we just take the first element
        if isinstance(data, tuple):
            data = data[0]
            
        # Forward pass
        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)
        
        # Calculate reconstruction loss
        reconstruction_loss = tf.reduce_mean(
            tf.reduce_sum(
                binary_crossentropy(data, reconstruction),
                axis=(1, 2)
            )
        ) * self.img_width * self.img_height
        
        # Calculate KL divergence loss
        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                axis=1
            )
        )
        
        # Total loss
        total_loss = reconstruction_loss + kl_loss
        
        # Update metrics
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

In [ ]:
LATENT_DIM=2

# Create the VAE model
vae = VAE(image_width=28, image_height=28, num_channels=1, latent_dim=LATENT_DIM)

# Compile the model (the loss is handled in the train_step method)
vae.compile(optimizer=Adam())

# Display model summary
vae.encoder.summary()
vae.decoder.summary()


In [ ]:
start = time()

## model fitting
history = vae.fit(X_train, epochs=100, batch_size=512, validation_split=0.2)

stop = time()
print('Time spent[s]: {:2f}'.format(stop -start))

In [ ]:
plot_history(history, width=14)


In [ ]:
## Model evaluate
vae.evaluate(X_test)

In [ ]:
# Visualization functions
def plot_latent_space(vae, x_test, figsize=15):
    # Display a 2D plot of the digit classes in the latent space
    z_mean, _, _ = vae.encoder.predict(x_test)
    plt.figure(figsize=(figsize, figsize))
    plt.scatter(z_mean[:, 0], z_mean[:, 1], c=y_test)
    plt.colorbar()
    plt.xlabel("z[0]")
    plt.ylabel("z[1]")
    plt.show()

def plot_latent_images(vae, scale=1.0, num_channels=1, n=15, digit_size=28):
    # Display a n*n 2D manifold of digits
    #scale = 1.0
    figure = zeros((digit_size * n, digit_size * n, num_channels))
    # linearly spaced coordinates corresponding to the 2D plot
    # of digit classes in the latent space
    grid_x = linspace(-scale, scale, n)
    grid_y = linspace(-scale, scale, n)[::-1]

    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z_sample = array([[xi, yi]])
            x_decoded = vae.decoder.predict(z_sample)
            digit = x_decoded[0].reshape(digit_size, digit_size, num_channels)
            figure[
                i * digit_size : (i + 1) * digit_size,
                j * digit_size : (j + 1) * digit_size,
            ] = digit

    plt.figure(figsize=(10, 10))
    start_range = digit_size // 2
    end_range = n * digit_size + start_range
    pixel_range = arange(start_range, end_range, digit_size)
    sample_range_x = round(grid_x, 1)
    sample_range_y = round(grid_y, 1)
    plt.xticks(pixel_range, sample_range_x)
    plt.yticks(pixel_range, sample_range_y)
    plt.xlabel("z[0]")
    plt.ylabel("z[1]")
    
    # matplotlib.pyplot.imshow() needs a 2D array, or a 3D array with the third dimension being of shape 3 or 4!
    # So reshape if necessary
    fig_shape = shape(figure)
    if fig_shape[2] == 1:
        figure = figure.reshape((fig_shape[0], fig_shape[1]))
    
    plt.imshow(figure)
    plt.show()

# Generate visualizations
if LATENT_DIM < 3:
    plot_latent_space(vae, x_test=X_test)
    plot_latent_images(vae, scale=4.0, n=20)
else:
    print('Latent dim is greater than 2.')



In [ ]:
## Compute prediction
prediction = vae.predict(X_test)
prediction.shape

In [ ]:
## Show some comparison 
seed(0)
indexes = randint(0, 10000, 18)

plt.figure(figsize=(14, 14))
for i, pos in enumerate(indexes):
    plt.subplot(6, 6, 2*i+1)
    plt.imshow(X_test[pos], cmap='gray')

    plt.subplot(6, 6, 2*i+2)
    plt.imshow(prediction[pos], cmap='gray')
plt.show()